# Train FATE Relation Detector on Kaggle

Clone the vn-av-forensics repository and attach an exported reviewed vn-av-relations-v2 generated bundle. Enable GPU and Internet. See the root README.md for Vietnamese instructions. This notebook uses the same CLI as the terminal.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess

REPO_URL = 'https://github.com/linhxm/vn-av-forensics.git'
REPO_ROOT = Path('/kaggle/working/vn-av-forensics')
ROOT = REPO_ROOT / 'vn-av-forensics-training'
if not REPO_ROOT.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)
os.chdir(ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.', '-r', 'environments/fate.txt'], check=True)
# Kaggle ships torchao 0.10, which PEFT 0.18 rejects even though this adapter does not use TorchAO.
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)


Restart the kernel after installation if Torch was already imported, then continue below.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import yaml

ROOT = Path('/kaggle/working/vn-av-forensics/vn-av-forensics-training')
os.chdir(ROOT)
# Set this to your exported dataset folder, possibly under /kaggle/input.
# Replace YOUR_DATASET with the Kaggle dataset slug shown under Input.
DATASET = Path('/kaggle/input/YOUR_DATASET/relations_two_head_v1')
assert DATASET.is_dir(), f'Missing Kaggle dataset: {DATASET}'
config_path = Path('configs/relations.yaml')
cfg = yaml.safe_load(config_path.read_text())
cfg['dataset'] = str(DATASET)
MANIFEST = 'manifest-reviewed.jsonl'  # Use manifest-two-heads.jsonl after migrating old labels.
cfg['dataset_manifest'] = MANIFEST
config_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')

def cli(*args):
    subprocess.run([sys.executable, '-m', 'vn_av_training', *args], check=True)

cli('validate')
cli('import')
cli('audit')


In [ ]:
cli('setup')
cli('doctor', '--load')


In [ ]:
cli('prepare')


In [ ]:
# One frozen backbone. First 3 epochs: timing warmup; remaining epochs: joint two-head training.
# After interruption: cli('train', '--resume') only if last.pt exists.
cli('train')


In [ ]:
cli('evaluate', '--split', 'test')
cli('checkpoint-info')


In [ ]:
# Analyze a clean held-out clip from the actual imported manifest.
import json
cfg = yaml.safe_load(Path('configs/relations.yaml').read_text())
rows = [json.loads(line) for line in Path(cfg['manifest']).read_text().splitlines() if line.strip()]
video = next(row['video'] for row in rows if row['split'] == 'test' and row.get('generation', {}).get('edit', {}).get('kind') == 'clean')
cli('analyze', '--video', video, '--output', 'outputs/demo')


In [ ]:
import zipfile

archive = Path('/kaggle/working/vn-av-forensics-artifacts.zip')
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for folder in ('runs', 'datasets/manifests', 'configs'):
        for path in (ROOT / folder).rglob('*'):
            if path.is_file():
                z.write(path, path.relative_to(ROOT))
print(f'Artifact ready: {archive}')


Download vn-av-forensics-artifacts.zip from Kaggle Output. For reuse, also persist cache/fate-two-head-v1, runs/fate-two-head-v1, datasets/manifests and configs, plus the separate generated dataset. Both required heads need positive/negative reviewed supervision before training; a trained head may still fail validation quality checks. Inspect checkpoint-info and validation-report.json. Synthetic-fixture tests do not establish detector accuracy.
